# Stage 4: embedding model evaluation

This notebook compares four local embedding configurations on ErgoStudy's development queries. Model selection uses only 64 development queries. The 32 final-test query IDs remain sealed for a later stage. Reusable loading, formatting, ranking, and metric logic lives in `src/`.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
split = json.loads((ROOT / 'data/processed/retrieval_eval_split.json').read_text(encoding='utf-8'))
print('Development queries:', len(split['development_query_ids']))
print('Sealed final-test queries:', len(split['final_test_query_ids']))
print('Final-test metrics computed in Stage 4: False')

## Candidate formatting

MiniLM and plain BGE use the original text. Instruction BGE adds its retrieval instruction only to queries. E5 adds `query:` to queries and `passage:` to corpus documents. Every output vector is float32 and normalized before direct cosine ranking.

In [ ]:
from src.embedding_models import load_embedding_configurations

settings, configurations = load_embedding_configurations(ROOT / 'config/embedding_models.json')
for item in configurations:
    print(item.config_id, '| query:', repr(item.query_prefix), '| document:', repr(item.document_prefix))

## Development results

The benchmark script measures direct normalized cosine search across the same 477 records for every configuration. This table is loaded from the reproducible output rather than downloading models again.

In [ ]:
summary = json.loads((ROOT / 'data/processed/embedding_benchmark_summary.json').read_text(encoding='utf-8'))
columns = ('config_id', 'recall_at_5', 'mrr_at_10', 'ndcg_at_10', 'average_query_latency_ms', 'document_embedding_time_seconds')
print(' | '.join(columns))
for result in summary['results']:
    overall = result['metrics']['overall']
    values = [result['config_id'], overall['recall_at_5'], overall['mrr_at_10'], overall['ndcg_at_10'], result['average_query_latency_ms'], result['document_embedding_time_seconds']]
    print(' | '.join(str(value) for value in values))

## Latency, resources, and truncation

Latency is measured per development query after one warm-up. GPU peak memory and safely measurable Hugging Face cache size are recorded when available. Token lengths are checked before encoding, and the benchmark stops instead of silently truncating a corpus record.

In [ ]:
for result in summary['results']:
    print(result['config_id'], {
        'device': result['device'],
        'median_ms': result['median_query_latency_ms'],
        'p95_ms': result['p95_query_latency_ms'],
        'peak_gpu_bytes': result['peak_gpu_memory_bytes'],
        'cache_bytes': result['cache_size_bytes'],
        'truncation_count': result['document_token_report']['truncation_count'],
    })

## Qualitative audit and decision

Each candidate stores three development examples for exact names, aliases, ambiguous wording, unseen wording, topics, study strategies, and sensor or safety cases. Criteria-only queries remain human judgments and are not converted into a text-matching score.

In [ ]:
selected_id = summary['selection_decision']['selected_config_id']
selected_result = next(result for result in summary['results'] if result['config_id'] == selected_id)
for category, examples in selected_result['qualitative_audit'].items():
    print('\n', category)
    for example in examples:
        print(example['query_id'], example['query'])
        print('expected IDs:', example['expected_record_ids'])
        print('criteria:', example['relevance_criteria'])
        for row in example['top_5']:
            print(row['record_id'], '|', row['title'], '|', row['document_family'], '|', row['subject_family'], '|', round(row['cosine_score'], 4))
        print(example['assessment'])
print('\nSelected:', selected_id)
print(summary['selection_decision']['reason'])